# comparison

In [ ]:
import os, sys

# Drop any cached combra.metrics modules so the lazy_loader stub doesn't hand
# back a pre-edit `compare_folders`. We must clear the parent `combra.metrics`
# package too, otherwise its cached attribute survives a child reload.
for m in [k for k in list(sys.modules) if k == 'combra.metrics' or k.startswith('combra.metrics.')]:
    del sys.modules[m]

from combra.metrics.compare import compare_folders

# Shared canonical ethalon — same parquet 4_grid_plot reads (256x256, all steps).
ethalon_path = './grid_results/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360_msl5/angles_n360.parquet'

# Explicit parquet file paths (not folders) so compare_folders processes exactly
# one file per row — avoids duplicates when a folder contains multiple sample sizes.
GRID = './grid_results'
folder_paths = [
    # The N10000 file is what 4_grid_plot's diffit res=256 row uses; this row
    # MUST numerically match `diffit res=256 N=1000` in 4_grid_plot.
    f'{GRID}/00017-diffit-256-gpus2-batch192_N10000_msl5/angles_n1000.parquet',

    # kimg snapshots — regenerated with current combra code.
    f'{GRID}/00017-diffit-256-gpus2-batch192_kimg_004435_msl5/angles_n1000.parquet',
    f'{GRID}/00017-diffit-256-gpus2-batch192_kimg_006451_msl5/angles_n1000.parquet',
    f'{GRID}/00017-diffit-256-gpus2-batch192_kimg_008064_msl5/angles_n1000.parquet',
    f'{GRID}/00017-diffit-256-gpus2-batch192_kimg_012096_msl5/angles_n1000.parquet',
    f'{GRID}/00017-diffit-256-gpus2-batch192_kimg_016128_msl5/angles_n1000.parquet',
]

# Generated parquet uses class_0/1/2; ethalon uses class_Ultra_Co*. Map by training-class index.
class_map = {
    'class_0': 'class_Ultra_Co11',
    'class_1': 'class_Ultra_Co25',
    'class_2': 'class_Ultra_Co6_2',
}

# Set False to suppress the per-row table print from compare_folders.
VERBOSE = True

# Per-checkpoint FID from the training run's stats.jsonl. Eval rows carry both
# 'FID' and 'kimg'; key by zero-padded floor(kimg) so it matches the kimg_tag
# prefix compare_folders derives from the folder name (e.g. '004435').
import json
STATS_PATH = '../../DiffiT-v2/training-runs/00017-diffit-256-gpus2-batch192/stats.jsonl'
fid_by_kimg = {}
with open(STATS_PATH) as fh:
    for line in fh:
        rec = json.loads(line)
        if 'FID' in rec and 'kimg' in rec:
            fid_by_kimg[f"{int(rec['kimg']):06d}"] = float(rec['FID'])

# wdist is now angle-space EMD in degrees; coef=1 prints raw degrees.
res = compare_folders(folder_paths, ethalon_path, class_map=class_map, steps=[2], coef=1, verbose=VERBOSE, fid_by_kimg=fid_by_kimg)